# Results — AAVE V3 WETH utilization predictor

Evaluates the calibrated logistic regression's out-of-sample predictions and
the resulting "supply when p_increase > 0.6" strategy:

1. **Classification quality** — AUC, PR, Brier, log-loss, reliability diagram.
2. **Strategy PnL** — equity curves vs `always_supplied`, `random_matched_freq`,
   `naive_dU_lag1`.
3. **Threshold sweep** — APY uplift across `p_in ∈ [0.50, 0.80]`.
4. **Regime split** — performance bucketed by realized-vol quartile.
5. **Sanity checks** — shuffled-label test (AUC must collapse), look-ahead audit.

Run order:
1. `python -m model.train`
2. `python -m backtest.simulator`
3. open this notebook.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

import config
from model.evaluate import (
    classification_metrics,
    reliability_curve,
    strategy_summary,
)
from backtest.simulator import simulate, baseline_always, baseline_random, baseline_naive_du

sns.set_theme(style="whitegrid", context="notebook")

In [ ]:
preds = pd.read_parquet(config.CACHE_DIR / "model_predictions.parquet")
meta = pd.read_parquet(config.CACHE_DIR / "backtest_meta.parquet")
results = pd.read_parquet(config.CACHE_DIR / "backtest_results.parquet")

print(f"OOS days: {len(preds):,}  ({preds.index[0].date()} → {preds.index[-1].date()})")
print(f"y class balance: {preds['y_true'].mean():.3f}")
print(f"fire rate at p>{config.P_ENTER}: {preds['signal'].mean():.1%}")

## Classification metrics

A few numbers we care about:
- **AUC** for ranking quality.
- **Brier** for calibration. Lower is better; ~0.25 is the no-skill baseline.
- **Precision @ threshold** — when the model fires, how often is U actually higher 7d later?
- **Reliability diagram** — predicted vs observed frequency by bin. The 0.6 bin is the one that drives the strategy.

In [ ]:
metrics = classification_metrics(preds["y_true"], preds["p_increase"], threshold=config.P_ENTER)
pd.Series(metrics).to_frame("value")

In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve

fpr, tpr, _ = roc_curve(preds["y_true"], preds["p_increase"])
prec, rec, _ = precision_recall_curve(preds["y_true"], preds["p_increase"])

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(fpr, tpr, lw=1.5, label=f"AUC = {metrics['auc']:.3f}")
axes[0].plot([0, 1], [0, 1], "--", color="grey")
axes[0].set(xlabel="FPR", ylabel="TPR", title="ROC")
axes[0].legend()

axes[1].plot(rec, prec, lw=1.5, label=f"AP = {metrics['ap']:.3f}")
axes[1].axhline(metrics["base_rate"], ls="--", color="grey", label=f"base rate = {metrics['base_rate']:.2f}")
axes[1].set(xlabel="Recall", ylabel="Precision", title="Precision-Recall")
axes[1].legend()

rel = reliability_curve(preds["y_true"], preds["p_increase"], n_bins=10)
axes[2].plot(rel["p_mean"], rel["y_mean"], "o-", lw=1.5)
axes[2].plot([0, 1], [0, 1], "--", color="grey")
axes[2].axvline(config.P_ENTER, ls=":", color="red", label=f"threshold {config.P_ENTER}")
axes[2].set(xlabel="Mean predicted p", ylabel="Empirical frequency", title="Reliability")
axes[2].legend()

plt.tight_layout()

## Equity curves vs baselines

Three baselines:
- `always` — always supplied (the honest hurdle).
- `random` — random fire schedule with the same fire rate as the model.
- `naive` — supply if yesterday's ΔU > 0.

If `model` doesn't beat `always` net of gas, the ML didn't add anything.

In [ ]:
eq_cols = [c for c in results.columns if c.endswith("_equity")]
eq = results[eq_cols].copy()
eq.columns = [c.replace("_equity", "") for c in eq_cols]

summary = pd.DataFrame([strategy_summary(eq[c], c) for c in eq.columns]).set_index("label")
summary.style.format({"apy": "{:.2%}", "sharpe": "{:.2f}", "max_dd": "{:.2%}", "final_equity": "${:,.0f}"})

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
for col in eq.columns:
    ax.plot(eq.index, eq[col] / eq[col].iloc[0], label=col, lw=1.3)

# Shade the days the model is in-position
pos = results["model_position"].astype(int)
ymin, ymax = ax.get_ylim()
ax.fill_between(pos.index, ymin, ymax, where=pos == 1, color="C0", alpha=0.05, step="post")

ax.set(ylabel="Equity (normalized to start)", title="Strategy equity curves (net of gas)")
ax.legend(loc="upper left")
plt.tight_layout()

## Threshold sweep

Re-runs the model strategy for entry thresholds in [0.50, 0.80] holding
`P_EXIT = P_ENTER - 0.20` and the same min-hold. If the curve has a sharp
peak right at 0.6, that's overfit — we want a broad plateau.

In [ ]:
thr_grid = np.linspace(0.50, 0.80, 13)
sweep = []
for thr in thr_grid:
    res = simulate(
        preds["p_increase"],
        meta.loc[preds.index],
        label=f"thr_{thr:.2f}",
        p_enter=thr,
        p_exit=max(0.05, thr - 0.20),
    )
    summ = strategy_summary(res.equity, res.label)
    summ["threshold"] = thr
    summ["fire_rate"] = float((preds["p_increase"] > thr).mean())
    sweep.append(summ)

sweep_df = pd.DataFrame(sweep)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(sweep_df["threshold"], sweep_df["apy"] * 100, "o-", label="strategy APY (%)")
always_apy = strategy_summary(eq["always"], "always")["apy"] * 100
ax.axhline(always_apy, ls="--", color="grey", label=f"always-supplied = {always_apy:.2f}%")
ax.axvline(config.P_ENTER, ls=":", color="red", label=f"chosen threshold {config.P_ENTER}")
ax.set(xlabel="p_enter", ylabel="APY (%)", title="Threshold sweep")
ax.legend()
plt.tight_layout()
sweep_df

## Regime split

Bucket OOS days into realized-vol quartiles and look at the model's hit-rate
in each. A model that only works in calm regimes is fine — but you need to know.

In [ ]:
from features.builders import build_dataset

X, y, _ = build_dataset()
yz = X["yz_vol"].reindex(preds.index)
quartile = pd.qcut(yz, 4, labels=["Q1 low vol", "Q2", "Q3", "Q4 high vol"])

regime = (
    preds.assign(yz_q=quartile)
    .groupby("yz_q", observed=True)
    .apply(
        lambda g: pd.Series(
            {
                "n": len(g),
                "base_rate": g["y_true"].mean(),
                "fire_rate": (g["p_increase"] > config.P_ENTER).mean(),
                "precision": g.loc[g["p_increase"] > config.P_ENTER, "y_true"].mean(),
                "auc": (
                    float("nan")
                    if g["y_true"].nunique() < 2
                    else __import__("sklearn.metrics", fromlist=["roc_auc_score"]).roc_auc_score(
                        g["y_true"], g["p_increase"]
                    )
                ),
            }
        )
    )
)
regime

## Sanity check — shuffled-label test

Re-run a single fit on shuffled labels. If the AUC is meaningfully above 0.5,
there's a leak in the feature pipeline.

In [ ]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegressionCV
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

rng = np.random.default_rng(0)
y_shuf = pd.Series(rng.permutation(y.to_numpy()), index=y.index)

split = int(len(X) * 0.7)
inner_cv = TimeSeriesSplit(n_splits=3, gap=config.EMBARGO_DAYS)
pipe = Pipeline(
    steps=[
        ("scale", StandardScaler()),
        (
            "clf",
            LogisticRegressionCV(
                Cs=5,
                cv=inner_cv,
                penalty="l2",
                class_weight="balanced",
                max_iter=2000,
            ),
        ),
    ]
)
est = CalibratedClassifierCV(pipe, method="isotonic", cv=inner_cv)
est.fit(X.iloc[:split].to_numpy(), y_shuf.iloc[:split].to_numpy())
p_shuf = est.predict_proba(X.iloc[split:].to_numpy())[:, 1]
auc_shuf = roc_auc_score(y_shuf.iloc[split:].to_numpy(), p_shuf)
print(f"shuffled-label AUC = {auc_shuf:.3f}  (should be ≈ 0.5)")